# Day 4-03｜シュートイベント、リリースフレーム、2 次元の打ち出し角度

[Colab で開く](https://colab.research.google.com/github/henry753951/basketball-hackathon-course/blob/main/day4/d4_03_ball_tracking_and_release_point_lab_ja.ipynb)

[English](d4_03_ball_tracking_and_release_point_lab_en.ipynb) | [Japanese](d4_03_ball_tracking_and_release_point_lab_ja.ipynb) | [Traditional Chinese](d4_03_ball_tracking_and_release_point_lab.ipynb)

> Python によるバスケットボール映像分析  
> シュート候補、選手の姿勢、球の軌跡を組み合わせ、各シュートの
> リリースフレームと画像平面上の打ち出し角度を推定する。規則は Notebook 内に定義し、読んで変更できるようにする。

## 学習目標
- 高い `shot_score` が続くフレームを、シュート候補イベントにまとめる。
- 全ボール候補を残し、選手の手首付近からイベントごとの軌跡を作る。
- 球が手首から離れ、上昇し始める位置からリリースフレームを推定する。
- 離手直後の球位置から 2 次元の打ち出し角度を求め、同時刻の関節角度を読む。
- Notebook 内のしきい値、軌跡規則、角度関数を変更し、結果の変化を見る。
- イベント表、リリース前後の図、重畳動画の三段階で目視確認する。

## 完了の目安
- イベント数が映像内の実際のシュート数とおおむね一致する。
- 各軌跡がシュートする手の近くから始まり、離手後に実際の球をたどる。
- 確認図が「保持 -> 離手 -> 上昇」の妥当な順序を示す。
- `_shot_segments`、`_event_ball_path`、`_launch_angle` の役割を説明できる。
- 目視確認を通過したイベントだけで `launch_angle_deg` と関節角度を解釈する。

## 出力
- `assets/results/d4_03_shot_events.csv`
- `assets/results/d4_03_event_ball_track.csv`
- `assets/results/d4_03_shot_angle_plot.png`
- `assets/results/d4_03_shot_event_check.png`
- `assets/results/d4_03_shot_event_preview.mp4`


## Setup｜実行環境と Day 4-02 の出力を確認する

### Colab のランタイム設定

実行前に Colab の **ランタイム → ランタイムのタイプを変更** を開き、ハードウェア アクセラレータを
**T4 GPU** にする。別の NVIDIA GPU も利用できる。**TPU は選択しない**。
本教材の Ultralytics／PyTorch は TPU を直接利用せず、CPU での実行になるためである。

Colab では先に `init_colab_ja.ipynb` を実行する。次のセルは Drive、教材ルート、モデルを確認する。
`results directory` が Google Drive にあることを確認する。Day 4-03 はここから
`d4_02_pose_angles.csv` を読み、イベント表、確認図、動画も同じ場所に保存する。

初期設定では先頭 180 フレームを処理する。`launch_angle_deg` は **2 次元の画像平面上の推定値**であり、カメラの高さ、
撮影角度、透視、球の画面外への移動に影響される。固定した撮影位置での相対比較に使い、実際の 3 次元角度とはみなさない。


In [1]:
import os
os.environ["BASKETBALL_COURSE_LANGUAGE"] = "ja"

from pathlib import Path
import subprocess
import sys

DRIVE_MOUNTED = False
if "google.colab" in sys.modules:
    from google.colab import drive

    try:
        drive.mount("/content/drive")
        DRIVE_MOUNTED = True
    except NotImplementedError:
        print("この Colab ランタイムでは Drive をマウントできないため、ローカルの /content を使います。")

COURSE_ROOT_HINT = (
    Path("/content/drive/MyDrive/basketball_hackathon/course")
    if DRIVE_MOUNTED
    else next(
        (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
         if (p / "src" / "course_setup.py").exists()),
        Path("/content/basketball_hackathon/course"),
    )
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo, prepare_day4_workspace  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT, mount_drive=False)
DAY4_MODELS, RESULTS = prepare_day4_workspace(COURSE_ROOT)
print("Day 4 models:", {name: str(path) for name, path in DAY4_MODELS.items()})
print("results directory:", RESULTS)


教材ルート： /Users/henry753951/Developer/sandbox/basketball-hackathon-course
素材フォルダ： /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets
共通モジュール： /Users/henry753951/Developer/sandbox/basketball-hackathon-course/src
Day 4 models: {'ball_rim': '/Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/ball_rimV8.pt', 'shot_detection': '/Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/shot_detection.pt', 'yolo_pose': '/Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/pose/yolov8n-pose.pt'}
results directory: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results


## Step 1｜同じ動画、姿勢 CSV、ボールモデルを読む

Day 4-01 の変換済み動画、Day 4-02 の姿勢 CSV、
`ball_rimV8.pt` の三つを確認する。フレームを合わせるため、`MAX_FRAMES` は前の単元の処理範囲と一致させる。

CSV がないか空の場合は、不適切な入力から見かけだけ妥当な角度を作らないよう、処理を停止する。


In [2]:
import pandas as pd

# 汎用の描画・動画・モデル関数を読み込み、イベント規則は下で定義します。
from src.shot_event_utils import (
    plot_shot_event_analysis,
    render_shot_event_check_image,
    render_shot_event_overlay_video,
)
from src.video_utils import display_video_in_notebook, pick_first_converted_video
from src.yolo_utils import load_yolo_model, preferred_inference_device

video_path = pick_first_converted_video(COURSE_ROOT)
pose_csv = RESULTS / "d4_02_pose_angles.csv"
ball_model_path = DAY4_MODELS["ball_rim"]
DEVICE = preferred_inference_device()
MAX_FRAMES = 180

if not pose_csv.exists():
    raise FileNotFoundError(f"姿勢の結果がありません：{pose_csv}。先に Day 4-02 を完了してください。")

pose_df = pd.read_csv(pose_csv)
if pose_df.empty:
    raise RuntimeError("d4_02_pose_angles.csv が空です。Day 4-02 を再実行してください。")

print("using video:", video_path)
print("pose csv:", pose_csv)
print("ball model:", ball_model_path)
print("inference device:", DEVICE)
print("pose rows:", len(pose_df))


using video: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/converted/video_001.mp4
pose csv: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_02_pose_angles.csv
ball model: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/models/detectors/ball_rimV8.pt
inference device: mps
pose rows: 180


## Step 2A｜`shot_score` をシュート候補区間にする

まずボール候補の形式を定義し、続いて `_shot_segments` を定義する。この関数が使うのは Day 4-02 の
`shot_score` だけであり、球の位置は使わない。

- `threshold`：シュート信号が有効なフレームを決める。
- `max_gap`：短い信号低下を、同じシュート内にまとめる。
- `min_active_frames` と `min_peak`：短すぎる区間や弱い区間を除く。

ここで得るのは「シュートの可能性がある区間」で、離手フレームではない。


In [3]:
from dataclasses import dataclass
import math
from typing import Any, cast

import cv2
import numpy as np

@dataclass(frozen=True)
class BallCandidate:
    frame: int
    x: float
    y: float
    confidence: float
    bbox_xyxy: tuple[float, float, float, float]


def _row_float(row: pd.Series, key: str) -> float:
    return float(cast(Any, row.at[key]))


def _row_int(row: pd.Series, key: str) -> int:
    return int(round(_row_float(row, key)))


def _shot_segments(
    pose_df: pd.DataFrame,
    *,
    threshold: float = 0.30,
    min_peak: float = 0.40,
    max_gap: int = 4,
    min_active_frames: int = 3,
) -> list[dict[str, float | int]]:
    ordered = pose_df.sort_values("frame").copy()
    active = ordered.loc[ordered["shot_score"] >= threshold, ["frame", "shot_score"]]
    if active.empty:
        return []

    groups: list[list[tuple[int, float]]] = []
    current: list[tuple[int, float]] = []
    previous_frame: int | None = None
    for _, row in active.iterrows():
        frame = _row_int(row, "frame")
        score = _row_float(row, "shot_score")
        if previous_frame is None or frame - previous_frame <= max_gap:
            current.append((frame, score))
        else:
            groups.append(current)
            current = [(frame, score)]
        previous_frame = frame
    if current:
        groups.append(current)

    segments: list[dict[str, float | int]] = []
    for group in groups:
        if len(group) < min_active_frames:
            continue
        peak_frame, peak_score = max(group, key=lambda item: item[1])
        if peak_score < min_peak:
            continue
        segments.append(
            {
                "start_frame": group[0][0],
                "end_frame": group[-1][0],
                "peak_frame": peak_frame,
                "peak_score": peak_score,
                "active_frames": len(group),
            }
        )
    return segments


## Step 2B｜各フレームの全ボール候補を残す

`_ball_candidates_by_frame` は `ball_rimV8.pt` を実行する。最高信頼度の球だけには絞らない。複数の球が
同時に映ることがあり、実際に投げた球が最高スコアとは限らないためである。

各球の中心、矩形、信頼度をフレームごとに保存する。リング中心は複数の検出の中央値で求め、
球がおおむね左と右のどちらに動くべきかを判断する手掛かりにする。


In [4]:
def _ball_candidates_by_frame(
    video_path: str | Path,
    *,
    ball_model_path: str | Path,
    max_frames: int,
    conf: float,
    imgsz: int,
    device: str | int | None,
) -> tuple[dict[int, list[BallCandidate]], float | None]:
    model = load_yolo_model(ball_model_path)
    inference_device = device if device is not None else preferred_inference_device()
    names = getattr(model, "names", {}) or {}
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(video_path)

    candidates_by_frame: dict[int, list[BallCandidate]] = {}
    rim_centers: list[float] = []
    frame = 0
    while cap.isOpened() and frame < max_frames:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        result = model.predict(
            frame_bgr,
            conf=conf,
            imgsz=imgsz,
            device=inference_device,
            verbose=False,
        )[0]
        frame_candidates: list[BallCandidate] = []
        boxes = getattr(result, "boxes", None)
        if boxes is not None:
            for index in range(len(boxes)):
                class_id = int(boxes.cls[index].item())
                class_name = str(names.get(class_id, class_id)) if isinstance(names, dict) else str(class_id)
                xyxy = boxes.xyxy[index].detach().cpu().numpy()
                center_x = float((xyxy[0] + xyxy[2]) / 2.0)
                center_y = float((xyxy[1] + xyxy[3]) / 2.0)
                confidence = float(boxes.conf[index].item())
                if class_name.lower() == "rim":
                    rim_centers.append(center_x)
                if class_name.lower() != "ball":
                    continue
                frame_candidates.append(
                    BallCandidate(
                        frame=frame,
                        x=center_x,
                        y=center_y,
                        confidence=confidence,
                        bbox_xyxy=(
                            float(xyxy[0]),
                            float(xyxy[1]),
                            float(xyxy[2]),
                            float(xyxy[3]),
                        ),
                    )
                )
        candidates_by_frame[frame] = frame_candidates
        frame += 1
    cap.release()
    rim_x = float(np.median(rim_centers)) if rim_centers else None
    return candidates_by_frame, rim_x


## Step 2C｜手首の近くから球の軌跡をつなぐ

次の関数でイベントに対応する球を選ぶ。

1. `_wrist_distances`：候補球と左右の手首との距離を求める。
2. `_track_from_seed`：起点から先の位置を予測し、距離、方向、信頼度で次の点を選ぶ。
3. `_event_ball_path`：シュート開始付近で手首から 60 画素以内の起点を使い、最高スコアの全経路を選ぶ。
4. 手首から 40 画素以内にあった球が 40 画素の外へ移り、同時に上昇するとリリース候補にする。

画像サイズが大きく異なる場合は、まず 60 画素と 40 画素の距離しきい値を見直す。


In [5]:
def _wrist_distances(row: pd.Series, candidate: BallCandidate) -> dict[str, float]:
    distances: dict[str, float] = {}
    for side in ("left", "right"):
        wrist_x = _row_float(row, f"{side}_wrist_x")
        wrist_y = _row_float(row, f"{side}_wrist_y")
        distances[side] = math.hypot(candidate.x - wrist_x, candidate.y - wrist_y)
    return distances


def _track_from_seed(
    seed: BallCandidate,
    *,
    candidates_by_frame: dict[int, list[BallCandidate]],
    end_frame: int,
    direction: float,
) -> tuple[list[BallCandidate], float]:
    path = [seed]
    position = np.asarray([seed.x, seed.y], dtype=float)
    velocity = np.zeros(2, dtype=float)
    last_frame = seed.frame
    transition_error = 0.0

    for frame in range(seed.frame + 1, end_frame + 1):
        options = candidates_by_frame.get(frame, [])
        if not options:
            continue
        delta_frames = max(1, frame - last_frame)
        predicted = position + velocity * delta_frames
        ranked: list[tuple[float, float, BallCandidate]] = []
        for candidate in options:
            point = np.asarray([candidate.x, candidate.y], dtype=float)
            prediction_distance = float(np.linalg.norm(point - predicted))
            reverse_motion = max(0.0, -direction * (candidate.x - position[0]))
            cost = prediction_distance + reverse_motion * 2.0 - candidate.confidence * 12.0
            ranked.append((cost, prediction_distance, candidate))
        _, prediction_distance, candidate = min(ranked, key=lambda item: item[0])
        speed = float(np.linalg.norm(velocity))
        # 高速の球では見落とし後に探索範囲を広げる必要があります。ただし、
        # 上限を設けないと、離れた別の球につながる場合があります。
        gate = min(120.0, max(55.0, speed * delta_frames * 2.2 + 28.0))
        if prediction_distance > gate:
            continue
        point = np.asarray([candidate.x, candidate.y], dtype=float)
        observed_velocity = (point - position) / delta_frames
        velocity = observed_velocity if len(path) < 2 else velocity * 0.45 + observed_velocity * 0.55
        transition_error += prediction_distance
        position = point
        last_frame = frame
        path.append(candidate)

    if len(path) < 2:
        return path, -1e9
    start = np.asarray([path[0].x, path[0].y])
    finish = np.asarray([path[-1].x, path[-1].y])
    upward = max(0.0, start[1] - finish[1])
    toward_rim = max(0.0, direction * (finish[0] - start[0]))
    mean_confidence = float(np.mean([item.confidence for item in path]))
    score = (
        len(path) * 4.0
        + upward * 0.30
        + toward_rim * 0.18
        + mean_confidence * 12.0
        - transition_error / max(len(path), 1) * 0.08
    )
    return path, score


def _event_ball_path(
    segment: dict[str, float | int],
    *,
    pose_rows: dict[int, pd.Series],
    candidates_by_frame: dict[int, list[BallCandidate]],
    rim_x: float | None,
    max_frame: int,
) -> tuple[list[BallCandidate], int, int, str]:
    start = max(0, int(segment["start_frame"]) - 12)
    seed_end = min(max_frame, int(segment["start_frame"]) + 5)
    track_end = min(max_frame, int(segment["end_frame"]) + 14)
    seed_options: list[tuple[BallCandidate, float]] = []
    for frame in range(start, seed_end + 1):
        pose_row = pose_rows.get(frame)
        if pose_row is None:
            continue
        for candidate in candidates_by_frame.get(frame, []):
            nearest_wrist = min(_wrist_distances(pose_row, candidate).values())
            if nearest_wrist <= 60.0:
                seed_options.append((candidate, nearest_wrist))
    if not seed_options:
        raise RuntimeError(
            f"候補区間 {segment['start_frame']}-{segment['end_frame']} で手首付近の球がありません。"
        )

    best_path: list[BallCandidate] = []
    best_score = -1e9
    for seed, seed_distance in seed_options:
        direction = 1.0 if rim_x is None or rim_x >= seed.x else -1.0
        path, score = _track_from_seed(
            seed,
            candidates_by_frame=candidates_by_frame,
            end_frame=track_end,
            direction=direction,
        )
        score -= seed_distance * 0.12
        if score > best_score:
            best_path = path
            best_score = score

    if len(best_path) < 3:
        raise RuntimeError(
            f"候補区間 {segment['start_frame']}-{segment['end_frame']} の球軌跡の点が不足しています。"
        )

    release_index: int | None = None
    contact_index: int | None = None
    for index in range(len(best_path) - 1):
        current = best_path[index]
        following = best_path[index + 1]
        pose_row = pose_rows.get(current.frame)
        next_pose_row = pose_rows.get(following.frame)
        if pose_row is None or next_pose_row is None:
            continue
        current_distance = min(_wrist_distances(pose_row, current).values())
        next_distance = min(_wrist_distances(next_pose_row, following).values())
        moving_up = following.y < current.y - 2.0
        if current_distance <= 40.0 and next_distance > 40.0 and moving_up:
            contact_index = index
            release_index = index + 1
    if release_index is None or contact_index is None:
        release_index = min(
            range(len(best_path)),
            key=lambda index: abs(best_path[index].frame - int(segment["start_frame"])),
        )
        contact_index = max(0, release_index - 1)

    contact = best_path[contact_index]
    release = best_path[release_index]
    contact_row = pose_rows[contact.frame]
    shooting_side = min(
        _wrist_distances(contact_row, contact).items(),
        key=lambda item: item[1],
    )[0]
    return best_path, contact.frame, release.frame, shooting_side


## Step 2D｜離手直後の軌跡から 2 次元の打ち出し角度を求める

`_launch_angle` は離手直後の数点について、フレーム番号に対する `x` と `y` の線形近似から、
1 フレーム当たりの水平・垂直変位を求める。画像の y 軸は下向きなので、角度には `-velocity_y` を使う。

`_interpolate_event_path` は重畳動画の短い欠落だけを補う。角度の計算には検出点を使う。
`analyze_shot_events` は各関数を順に呼び、三つの DataFrame に整理する。分析規則はここで確認できる。


In [6]:
def _launch_angle(
    path: list[BallCandidate],
    *,
    release_frame: int,
) -> tuple[float, float, float]:
    points = [
        item
        for item in path
        if release_frame <= item.frame <= release_frame + 8
    ]
    if len(points) < 3:
        points = [item for item in path if item.frame >= release_frame][:5]
    if len(points) < 2:
        return float("nan"), float("nan"), float("nan")
    frames = np.asarray([item.frame for item in points], dtype=float)
    x_values = np.asarray([item.x for item in points], dtype=float)
    y_values = np.asarray([item.y for item in points], dtype=float)
    velocity_x = float(np.polyfit(frames, x_values, 1)[0])
    velocity_y = float(np.polyfit(frames, y_values, 1)[0])
    angle = math.degrees(math.atan2(-velocity_y, abs(velocity_x)))
    return angle, velocity_x, velocity_y


def _interpolate_event_path(
    path: list[BallCandidate],
    *,
    event_id: int,
    max_gap: int = 8,
) -> list[dict[str, float | int | str]]:
    rows: list[dict[str, float | int | str]] = []
    for index, current in enumerate(path):
        rows.append(
            {
                "event_id": event_id,
                "frame": current.frame,
                "x": current.x,
                "y": current.y,
                "confidence": current.confidence,
                "source": "detect",
            }
        )
        if index >= len(path) - 1:
            continue
        following = path[index + 1]
        gap = following.frame - current.frame
        if gap <= 1 or gap > max_gap:
            continue
        for step in range(1, gap):
            ratio = step / gap
            rows.append(
                {
                    "event_id": event_id,
                    "frame": current.frame + step,
                    "x": current.x + (following.x - current.x) * ratio,
                    "y": current.y + (following.y - current.y) * ratio,
                    "confidence": min(current.confidence, following.confidence),
                    "source": "interpolate",
                }
            )
    return rows


def analyze_shot_events(
    video_path: str | Path,
    pose_df: pd.DataFrame,
    *,
    ball_model_path: str | Path,
    max_frames: int = 180,
    ball_conf: float = 0.10,
    imgsz: int = 1280,
    device: str | int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """シュートスコア、選手の姿勢、球の動きからリリースイベントを求めます。"""
    frame_features = pose_df.loc[pose_df["frame"] < max_frames].copy()
    frame_features["shot_score_smooth"] = (
        frame_features["shot_score"].rolling(5, center=True, min_periods=1).mean()
    )
    segments = _shot_segments(frame_features)
    if not segments:
        raise RuntimeError("shot_detection.pt にしきい値を超える候補イベントがありません。")

    candidates_by_frame, rim_x = _ball_candidates_by_frame(
        video_path,
        ball_model_path=ball_model_path,
        max_frames=max_frames,
        conf=ball_conf,
        imgsz=imgsz,
        device=device,
    )
    pose_rows = {
        _row_int(row, "frame"): row
        for _, row in frame_features.sort_values("frame").iterrows()
    }
    event_rows: list[dict[str, float | int | str]] = []
    ball_rows: list[dict[str, float | int | str]] = []
    for event_index, segment in enumerate(segments, start=1):
        path, contact_frame, release_frame, shooting_side = _event_ball_path(
            segment,
            pose_rows=pose_rows,
            candidates_by_frame=candidates_by_frame,
            rim_x=rim_x,
            max_frame=max_frames - 1,
        )
        angle, velocity_x, velocity_y = _launch_angle(path, release_frame=release_frame)
        release_pose = pose_rows.get(release_frame, pose_rows[contact_frame])
        event_rows.append(
            {
                "event_id": event_index,
                **segment,
                "contact_frame": contact_frame,
                "release_frame": release_frame,
                "shooting_side": shooting_side,
                "release_elbow_angle": float(
                    release_pose[f"{shooting_side}_elbow_angle"]
                ),
                "release_shoulder_angle": float(
                    release_pose[f"{shooting_side}_shoulder_angle"]
                ),
                "release_knee_angle": float(
                    release_pose[f"{shooting_side}_knee_angle"]
                ),
                "launch_angle_deg": angle,
                "launch_velocity_x_px_per_frame": velocity_x,
                "launch_velocity_y_px_per_frame": velocity_y,
                "trajectory_detected_points": len(path),
                "rim_x": rim_x if rim_x is not None else float("nan"),
            }
        )
        ball_rows.extend(_interpolate_event_path(path, event_id=event_index))

    events_df = pd.DataFrame(event_rows)
    event_ball_df = (
        pd.DataFrame(ball_rows)
        .sort_values(["event_id", "frame", "source"])
        .drop_duplicates(["event_id", "frame"], keep="first")
        .reset_index(drop=True)
    )
    return frame_features, events_df, event_ball_df


## Step 2E｜イベント分析を実行し、中間データを保存する

ここで四つの関数群をつなぐ。`ball_conf = 0.10` は小さい球やぼけた球の候補を残す設定である。低信頼度の候補が
そのまま答えになるわけではなく、手首との距離、軌跡の連続性、移動方向も確認する。

三つの表にフレーム特徴、イベント概要、イベント別の球軌跡を保存する。先に CSV にしておけば、図の描き直しや
特定イベントの確認で、モデルの推論を再実行する必要がない。


In [7]:
# 三つの DataFrame にフレーム特徴、イベント概要、イベント別の球位置を保存します。
frame_features_df, events_df, event_ball_df = analyze_shot_events(
    video_path,
    pose_df,
    ball_model_path=ball_model_path,
    max_frames=MAX_FRAMES,
    ball_conf=0.10,
    imgsz=1280,
    device=DEVICE,
)

frame_features_csv = RESULTS / "d4_03_shot_frame_features.csv"
events_csv = RESULTS / "d4_03_shot_events.csv"
event_ball_csv = RESULTS / "d4_03_event_ball_track.csv"

# 中間データを保存し、図の描き直しで推論を再実行せずに済むようにします。
frame_features_df.to_csv(frame_features_csv, index=False)
events_df.to_csv(events_csv, index=False)
event_ball_df.to_csv(event_ball_csv, index=False)

print("saved:", frame_features_csv)
print("saved:", events_csv)
print("saved:", event_ball_csv)
events_df[[
    "event_id", "start_frame", "end_frame", "peak_frame",
    "release_frame", "shooting_side", "launch_angle_deg",
    "release_elbow_angle", "release_shoulder_angle", "release_knee_angle",
]].round(2)


saved: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_03_shot_frame_features.csv
saved: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_03_shot_events.csv
saved: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_03_event_ball_track.csv


,event_id,start_frame,end_frame,peak_frame,release_frame,shooting_side,launch_angle_deg,release_elbow_angle,release_shoulder_angle,release_knee_angle
0,1,43,51,51,43,right,47.74,165.17,126.70,159.38
1,2,147,168,157,149,right,41.30,150.97,117.79,158.06


## Step 3｜イベント表を読み、二つの確認図を描く

各フレーム項目の役割は異なる。`start_frame`～`end_frame` は候補区間、`peak_frame` は
`shot_score` の最大位置、`release_frame` は手首との関係から推定した離手時刻であり、一致する必要はない。

`plot_shot_event_analysis` はスコア、区間、角度を時系列で表示する。
`render_shot_event_check_image` は各イベントの離手 2 フレーム前、離手時、4 フレーム後を並べる。
赤い球マーカーが手の近くから離れ、実際の球路をたどるかを見る。別の球、リング、背景に移る場合は、
その角度を信頼できない値として扱う。


In [8]:
angle_plot_png = RESULTS / "d4_03_shot_angle_plot.png"
event_check_png = RESULTS / "d4_03_shot_event_check.png"

plot_shot_event_analysis(
    frame_features_df,
    events_df,
    output_path=angle_plot_png,
)
render_shot_event_check_image(
    video_path,
    frame_features_df,
    events_df,
    event_ball_df,
    event_check_png,
)

print("saved:", angle_plot_png)
print("saved:", event_check_png)


saved: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_03_shot_angle_plot.png
saved: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_03_shot_event_check.png


## Step 4｜重畳動画全体で時間的な連続性を確認する

静止画では離手前後、動画ではイベント全体を確認する。黄色の矩形がシュートする選手を囲むか、
白い骨格が同じ人物を追うか、赤い球が今回のシュートを追うか、離手時に `RELEASE` が出るかを順に見る。

四点が妥当な場合に、打ち出し角度と離手時の肘角度を比べる。選手間やシュート間ではカメラ位置、
画面の縮尺、撮影方向を固定する。そうしなければ、差が動作ではなくカメラに由来する可能性がある。


In [9]:
overlay_mp4 = RESULTS / "d4_03_shot_event_preview.mp4"

# 区間、選手の骨格、イベントの球、リリース情報を動画に重ねます。
render_shot_event_overlay_video(
    video_path,
    frame_features_df,
    events_df,
    event_ball_df,
    overlay_mp4,
    max_frames=MAX_FRAMES,
)

print("saved:", overlay_mp4)
display_video_in_notebook(overlay_mp4, width=820, muted=True, loop=True)


$ /opt/homebrew/bin/ffmpeg -y -i /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_03_shot_event_preview.mp4 -an -vcodec libx264 -pix_fmt yuv420p -movflags +faststart -preset veryfast -crf 23 /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_03_shot_event_preview.notebook.mp4


saved: /Users/henry753951/Developer/sandbox/basketball-hackathon-course/assets/results/d4_03_shot_event_preview.mp4


## Checks｜Day 4 の最終確認

- 実際のシュート数が `d4_03_shot_events.csv` のイベント数とおおむね一致する。
- 各軌跡がシュートする手の近くから始まり、別の球につながっていない。
- 離手前後の順序が妥当で、`RELEASE` が明らかに早すぎたり遅すぎたりしない。
- 離手付近の関節点が妥当な場合だけ、肘、肩、膝の角度を解釈する。
- 打ち出し角度は固定したカメラで比較し、2 次元の画像平面上の推定値と呼ぶ。

## `assets/results/` に保存されるファイル

- `d4_03_shot_frame_features.csv`：フレームごとのシュートスコアと姿勢角度。
- `d4_03_shot_events.csv`：各シュートの区間、離手フレーム、関節角度、打ち出し角度。
- `d4_03_event_ball_track.csv`：各イベントの球軌跡。
- `d4_03_shot_angle_plot.png`：イベントと角度の時系列図。
- `d4_03_shot_event_check.png`：離手前、離手時、離手後の確認図。
- `d4_03_shot_event_preview.mp4`：イベント全体の重畳動画。

## Day 4 の振り返り｜三つのモデルの使い道

| モデル | ファイル | 主な用途 | プロジェクトへの応用 |
| --- | --- | --- | --- |
| `ball_rimV8.pt` | `assets/models/detectors/ball_rimV8.pt` | ボールとリングの候補矩形 | 球軌跡、シュート数、球とリングの位置関係 |
| `shot_detection.pt` | `assets/models/detectors/shot_detection.pt` | シュート候補の人物と `shot_score` | シュート場面の抽出、選手の特定、イベント区間 |
| `yolov8n-pose.pt` | `assets/models/pose/yolov8n-pose.pt` | 17 個の COCO 人体キーポイント | 関節角度、動作の区分、姿勢特徴の比較 |

必要なモデルだけを読み込めばよい。シュート全体を分析する場合は、各出力を
「シュート候補 -> 選手の姿勢 -> イベント別の球軌跡 -> 離手」とつなぐ。重みはすべて `assets/models/` にあり、
`init_colab_ja.ipynb` で Google Drive に同期されるため、別途ダウンロードする必要はない。
